In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)

sns.set_theme(style="whitegrid")

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
BASE = "../data/raw/"

ball = pd.read_csv(BASE + "ball_by_ball.csv")
matches = pd.read_csv(BASE + "matches.csv")
competitions = pd.read_csv(BASE + "competitions.csv")
teams = pd.read_csv(BASE + "teams.csv")
playing_xi = pd.read_csv(BASE + "playing_eleven.csv")
player_mapping = pd.read_csv(
    BASE + "competition_team_player_mapping.csv"
)

print("All datasets loaded successfully!")

All datasets loaded successfully!


In [3]:
datasets = {
    "Ball by Ball": ball,
    "Matches": matches,
    "Competitions": competitions,
    "Teams": teams,
    "Playing XI": playing_xi,
    "Player Mapping": player_mapping
}

for name, df in datasets.items():
    print(name, ":", df.shape)

Ball by Ball : (200999, 33)
Matches : (883, 22)
Competitions : (141, 4)
Teams : (353, 3)
Playing XI : (28381, 9)
Player Mapping : (6413, 6)


In [4]:
for name, df in datasets.items():
    print("\n", name)
    print(df.info())


 Ball by Ball
<class 'pandas.DataFrame'>
RangeIndex: 200999 entries, 0 to 200998
Data columns (total 33 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   match_id             200999 non-null  int64  
 1   innings              200999 non-null  int64  
 2   overid               200999 non-null  int64  
 3   over_ball_num        200999 non-null  int64  
 4   innings_ball_num     200999 non-null  int64  
 5   batting_team_id      200999 non-null  int64  
 6   bowling_team_id      200999 non-null  int64  
 7   striker_id           200999 non-null  int64  
 8   striker_name         200999 non-null  str    
 9   non_striker_id       200999 non-null  int64  
 10  non_striker_name     200999 non-null  str    
 11  bowler_id            200999 non-null  int64  
 12  bowler_name          200999 non-null  str    
 13  batting_type         200999 non-null  str    
 14  bowling_type         200999 non-null  str    
 15  batting_order

In [5]:
for name, df in datasets.items():
    print("\n", name)
    print(df.isnull().sum()[df.isnull().sum() > 0])


 Ball by Ball
dismissal_type_id      190118
dismissal_type         190118
dismissed_player_id    190107
extras_type            189167
stroke                   6158
delivery_style            391
over_or_around            407
dtype: int64

 Matches
home_team_captain_id         44
away_team_captain_id         45
toss_win_team_id             15
toss_opted                   15
first_inn_revised_overs     858
second_inn_revised_overs    858
revised_target              860
win_team_id                  41
dtype: int64

 Competitions
Series([], dtype: int64)

 Teams
Series([], dtype: int64)

 Playing XI
batting_order    8678
is_captain       8667
is_wk            8667
dtype: int64

 Player Mapping
Series([], dtype: int64)


In [10]:
# Merge match and competition information
match_data = matches.merge(
    competitions,
    on="comp_id",
    how="left",
    validate="many_to_one"
)

# Convert dates
match_data["match_date"] = pd.to_datetime(
    match_data["match_date"],
    errors="coerce"
)

# Filter from 1 January 2022
match_data = match_data[
    match_data["match_date"] >= "2022-01-01"
]

# Identify IPL and T20I
ipl = match_data[
    match_data["comp_name"].str.contains(
        r"\bIPL\b", case=False, na=False
    )
]

t20i = match_data[
    match_data["match_format"] == "T20I"
]

# Combine the two competitions
primary_matches = pd.concat(
    [ipl, t20i]
).drop_duplicates(subset="match_id")

# Merge metadata with ball-by-ball data
primary_ball = ball.merge(
    primary_matches[
        [
            "match_id", "comp_id", "comp_name",
            "match_format", "season", "match_date",
            "venue_country", "revised_target",
            "first_inn_revised_overs",
            "second_inn_revised_overs"
        ]
    ],
    on="match_id",
    how="inner",
    validate="many_to_one"
)

print("IPL matches:", ipl["match_id"].nunique())
print("T20I matches:", t20i["match_id"].nunique())

print("Total primary matches:",
      primary_matches["match_id"].nunique())

print("Matches with ball data:",
      primary_ball["match_id"].nunique())

print("Total deliveries:", len(primary_ball))

IPL matches: 371
T20I matches: 482
Total primary matches: 853
Matches with ball data: 837
Total deliveries: 194014


In [7]:
main_innings = primary_ball[
    primary_ball["innings"].isin([1, 2])
].copy()

print("Main innings deliveries:", len(main_innings))

Main innings deliveries: 193988


In [8]:
print("Extras types:")
print(ball["extras_type"].value_counts(dropna=False))

print("\nDismissal types:")
print(ball["dismissal_type"].value_counts(dropna=False))

print("\nRuns distribution:")
print(ball["runs"].value_counts().sort_index())

print("\nExample wide deliveries:")
print(
    ball[ball["extras_type"] == "wide"][
        ["runs", "extras1_runs", "extras_2_runs",
         "is_wicket", "dismissal_type"]
    ].head(10)
)

print("\nExample no-ball deliveries:")
print(
    ball[ball["extras_type"] == "Overstep"][
        ["runs", "extras1_runs", "extras_2_runs",
         "is_wicket", "dismissal_type"]
    ].head(10)
)

Extras types:
extras_type
NaN         189167
wide          7714
legbyes       2748
Overstep       576
byes           529
other          264
Penalty          1
Name: count, dtype: int64

Dismissal types:
dismissal_type
NaN                      190118
Caught                     7557
Bowled                     1775
LBW                         709
RunOut                      600
Stump                       214
Hit Wicket                   17
Retire Out                    6
Obstracting the field         2
Obstructing the field         1
Name: count, dtype: int64

Runs distribution:
runs
0    79387
1    72267
2    12547
3      656
4    23303
5       25
6    12814
Name: count, dtype: int64

Example wide deliveries:
     runs  extras1_runs  extras_2_runs  is_wicket dismissal_type
39      0             1              0      False            NaN
43      0             1              0      False            NaN
44      0             1              0      False            NaN
278     0             

In [9]:
os.makedirs("../data/processed", exist_ok=True)

primary_ball.to_parquet(
    "../data/processed/primary_ball.parquet",
    index=False
)

primary_matches.to_csv(
    "../data/processed/primary_matches.csv",
    index=False
)

print("Primary datasets saved!")

Primary datasets saved!


In [11]:
# Reconstruct total runs
ball["total_delivery_runs"] = (
    ball["runs"]
    + ball["extras1_runs"]
    + ball["extras_2_runs"]
)

# Identify illegal deliveries
ball["is_wide"] = ball["extras_type"].eq("wide")

ball["is_no_ball"] = ball["extras_type"].eq("Overstep")

ball["is_legal_ball"] = ~(
    ball["is_wide"] | ball["is_no_ball"]
)

# Inspect unusual extras
print("Total batter runs:", ball["runs"].sum())

print("Total extra runs:",
      ball["extras1_runs"].sum()
      + ball["extras_2_runs"].sum())

print("Total team runs:", ball["total_delivery_runs"].sum())

print("Legal deliveries:", ball["is_legal_ball"].sum())

print("Illegal deliveries:",
      (~ball["is_legal_ball"]).sum())

print("\nAdditional extras records:")

print(
    ball[ball["extras_2_runs"] > 0][
        ["match_id", "runs", "extras_type",
         "extras1_runs", "extras_2_runs"]
    ].head(20)
)

Total batter runs: 269550
Total extra runs: 14707
Total team runs: 284257
Legal deliveries: 192709
Illegal deliveries: 8290

Additional extras records:
       match_id  runs extras_type  extras1_runs  extras_2_runs
1485      13523     0    Overstep             1              4
2588      13614     0       other             1              4
7607      14112     0    Overstep             1              4
11386     14127     0       other             1              4
18844     14159     0       other             1              1
24453     15796     0    Overstep             1              1
28010     16424     0    Overstep             1              4
36854     17555     0         NaN             0              1
47440     20186     0    Overstep             1              1
47803     20188     0       other             1              1
50553     20199     0    Overstep             1              1
50935     20201     0    Overstep             1              4
55141     20218     0    Over